In [14]:
import pandas as pd
import os
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.metrics import brier_score_loss

In [15]:
'''
Function to merge two dataframes
param df1: first dataframe
param df2: second dataframe
return: a single dataframe of the merged dataframes 
'''
def mergeDataframes(df1, df2):
    df = pd.concat([df1, df2])
    return df


'''
Function to merge tournament data for mens and womens and crop the data to match reg detail
params df1: dataframe of mens tournament data
params df2: dataframe of womens tournament data
return: dataframe of combined and cropped tournament data
'''
def mergeTournamentData(df1, df2):
    df1 = df1[df1['Season'] >= 2003].reset_index(drop=True)
    df2 = df2[df2['Season'] >= 2010].reset_index(drop=True)
    
    df = pd.concat([df1, df2])
    return df


'''
Function to split regular season detailed results into dataframes focused on outcome for one team
param df: regular season data
return: a dataframe where each team from a single row in reg data has its own row
'''
def regularDetailsFocus(df):
    RegWinners = pd.DataFrame()
    RegLossers = pd.DataFrame()

    # Establish new columns for that includes stats for one team
    columns = ['Season', 'TeamID', 'DayNum', 'Score', 'OppScore',
               'NumOT', 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
               'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF', 'OppFGM', 'OppFGA',
               'OppFGM3', 'OppFGA3', 'OppFTM', 'OppFTA', 'OppOR', 'OppDR', 'OppAst', 'OppTO',
               'OppStl', 'OppBlk', 'OppPF']

    # Split winners from regular season
    RegWinners[columns] = df[['Season', 'WTeamID', 'DayNum', 'WScore', 'LScore',
                              'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA',
                              'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA',
                              'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO',
                              'LStl', 'LBlk', 'LPF']]

    # Add wins and losses columns
    RegWinners['Win'] = 1
    RegWinners['Loss'] = 0

    # Split lossers from regular season
    RegLossers[columns] = df[['Season', 'LTeamID', 'DayNum', 'LScore', 'WScore',
                               'NumOT', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA',
                               'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF', 'WFGM', 'WFGA',
                               'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO',
                               'WStl', 'WBlk', 'WPF']]

    # Add wins and losses columns
    RegLossers['Win'] = 0
    RegLossers['Loss'] = 1

    # Combine all games into one dataframe
    AllRegDetail = pd.concat([RegWinners, RegLossers])
    
    return AllRegDetail


'''
Function to clean seed column
param seeds: Dataframe of historical seeds with column 'Seed'
return: A dataframe with the 'Seed' column converted to int
'''

def cleanSeed(seeds):
    seeds['Seed'] = seeds['Seed'].str.extract(r'(\d+)').astype(int)
    return seeds


'''
Function to join two Dataframes on 'TeamID'
param seeds: Dataframe of historical seeds with column 'Seed'
param tourny: Dataframe of compact tournament data with columns 'WTeamID' and 'LTeamID'
return: a single dataframe of the joined Dataframes
'''

def joinSeeds(seeds, tourny):
    seeds = seeds.set_index(['Season','TeamID'])

    tourny = tourny.join(
        seeds.rename(columns={'Seed':'WSeed'}),
        on=['Season','WTeamID']
    )

    tourny = tourny.join(
        seeds.rename(columns={'Seed':'LSeed'}),
        on=['Season','LTeamID']
    )
    return tourny


'''
Function to create the features from the regular season data at a single-game level
param df: dataframe of regular season data
return: dataframe of input features at a single-game level
'''
def createFeatures(df):
    RegSeasonFeatures = pd.DataFrame()
    RegSeasonFeatures[['Season', 'TeamID', 'DayNum']] = df[['Season', 'TeamID', 'DayNum']]

    RegSeasonFeatures['PointRatio'] = df['Score'] / df['OppScore']  # Points ratio
    RegSeasonFeatures['MOV'] = df['Score'] - df['OppScore']  # Margin of victory
    RegSeasonFeatures['TORatio'] = df['TO'] / df['OppTO']  # Turnover ratio
    RegSeasonFeatures['FGM%'] = df['FGM'] / df['FGA']  # Scoring efficiency
    RegSeasonFeatures['FG3%M'] = df['FGM3'] / df['FGA3']  # 3-Point efficiency
    RegSeasonFeatures['FGA3%'] = df['FGA3'] / df['FGA']  # 3-Point attempt rate
    RegSeasonFeatures['FTM%'] = df['FTM'] / df['FTA']  # Free throw makes %
    RegSeasonFeatures['OppFTM%'] = df['OppFTM'] / df['OppFTA']  # Opponent free throw makes %
    RegSeasonFeatures['FTR'] = df['FTA'] / df['FGA']  # Free throw attempt rate
    RegSeasonFeatures['OppFTR'] = df['OppFTA'] / df['OppFGA']  # Opponent free throw attempt rate
    RegSeasonFeatures['ORRatio'] = df['OR'] / (df['OR'] + df['OppDR'])  # Offensive rebound ratio
    RegSeasonFeatures['DRRatio'] = df['DR'] / (df['DR'] + df['OppOR'])  # Defensive rebound ratio
    
    # New for 2026 (More advanced!)
    RegSeasonFeatures['NumPos'] = df['FGA'] - df['OR'] + df['TO'] + (0.44 * df['FTA'])  # ROUGH estimation of positions
    RegSeasonFeatures['OffEff'] = df['Score'] / RegSeasonFeatures['NumPos']  # Offensive efficiency
    RegSeasonFeatures['DefEff'] = df['OppScore'] / RegSeasonFeatures['NumPos']  # Defensive efficiency
    RegSeasonFeatures['NetEff'] = RegSeasonFeatures['OffEff'] - RegSeasonFeatures['DefEff']  # Net efficiency
    RegSeasonFeatures['TO%'] = df['TO'] / RegSeasonFeatures['NumPos']  # Turnover %
    RegSeasonFeatures['Ast%'] = df['Ast'] / df['FGM']  # Assist percentage
    RegSeasonFeatures['AstTORatio'] = df['Ast'] / df['TO']  # Assist to turnover ratio
    RegSeasonFeatures['AstRatio'] = df['Ast'] / df['OppAst']  # Assist ratio
    RegSeasonFeatures['OR%'] = df['OR'] / (df['FGA'] - df['FGM'])  # Offensive rebound %
    RegSeasonFeatures['DR%'] = df['DR'] / (df['OppFGA'] - df['OppFGM'])  # Defensive rebound %
    RegSeasonFeatures['EffFG%'] = (df['FGM'] + (0.5 * df['FGM3'])) / df['FGA']  # Effective field goal %
    RegSeasonFeatures['TS%'] = df['Score'] / (2 * (df['FGA'] + (0.44 * df['FTA'])))  # True shot %
    
    return RegSeasonFeatures

In [16]:
# Mens data import
mRegDetail = pd.read_csv('data/men/MRegularSeasonDetailedResults.csv')
mTournCompact = pd.read_csv('data/men/MNCAATourneyCompactResults.csv')
mTournSeeds = pd.read_csv('data/men/MNCAATourneySeeds.csv')
mNames = pd.read_csv('data/men/MTeamSpellings.csv')

# Womens data import
wRegDetail = pd.read_csv('data/women/WRegularSeasonDetailedResults.csv')
wTournCompact = pd.read_csv('data/women/WNCAATourneyCompactResults.csv')
wTournSeeds = pd.read_csv('data/women/WNCAATourneySeeds.csv')
wNames = pd.read_csv('data/women/WTeamSpellings.csv')

# Clean and merge seeds with tournament results
mCleanSeeds = cleanSeed(mTournSeeds.copy())
wCleanSeeds = cleanSeed(wTournSeeds.copy())
mFullTourn = joinSeeds(mCleanSeeds, mTournCompact)
wFullTourn = joinSeeds(wCleanSeeds, wTournCompact)

# Combined data
regDetail = mergeDataframes(mRegDetail, wRegDetail)
compactTourn = mergeTournamentData(mFullTourn, wFullTourn)
names = mergeDataframes(mNames, wNames)

# Split regular season detailed results into dataframes focused on outcome for one team
AllRegDetail = regularDetailsFocus(regDetail)

# Create single-game features
features = createFeatures(AllRegDetail)

In [17]:
AllRegDetail

,Season,TeamID,DayNum,Score,OppScore,NumOT,FGM,FGA,FGM3,FGA3,...,OppFTA,OppOR,OppDR,OppAst,OppTO,OppStl,OppBlk,OppPF,Win,Loss
0,2003,1104,10,68,62,0,27,58,3,14,...,22,10,22,8,18,9,2,20,1,0
1,2003,1272,10,70,63,0,26,62,8,20,...,20,20,25,7,12,8,6,16,1,0
2,2003,1266,11,73,61,0,24,58,8,18,...,23,31,22,9,12,2,5,23,1,0
3,2003,1296,11,56,50,0,18,38,3,9,...,15,17,20,9,19,4,3,23,1,0
4,2003,1400,11,77,71,0,30,61,6,14,...,27,21,15,12,10,7,1,14,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86768,2026,3397,118,77,87,0,31,68,6,21,...,6,6,21,19,5,3,14,14,0,1
86769,2026,3438,118,82,83,0,30,60,4,15,...,22,9,21,18,7,5,9,24,0,1
86770,2026,3332,118,69,70,0,26,53,4,16,...,12,10,19,13,7,2,11,17,0,1
86771,2026,3153,118,60,118,0,22,53,2,11,...,30,14,27,26,11,1,4,20,0,1


In [18]:
features

,Season,TeamID,DayNum,PointRatio,MOV,TORatio,FGM%,FG3%M,FGA3%,FTM%,...,DefEff,NetEff,TO%,Ast%,AstTORatio,AstRatio,OR%,DR%,EffFG%,TS%
0,2003,1104,10,1.096774,6,1.277778,0.465517,0.214286,0.241379,0.611111,...,0.827549,0.080085,0.306994,0.481481,0.565217,1.625000,0.451613,0.774194,0.491379,0.515777
1,2003,1272,10,1.111111,7,1.083333,0.419355,0.400000,0.322581,0.526316,...,0.921592,0.102399,0.190170,0.615385,1.230769,2.285714,0.416667,0.651163,0.483871,0.497442
2,2003,1266,11,1.196721,12,0.833333,0.413793,0.444444,0.310345,0.586207,...,0.956713,0.188206,0.156838,0.625000,1.500000,1.666667,0.500000,0.509804,0.482759,0.515828
3,2003,1296,11,1.120000,6,0.631579,0.473684,0.333333,0.236842,0.548387,...,0.867453,0.104094,0.208189,0.611111,0.916667,1.222222,0.300000,0.612903,0.513158,0.542215
4,2003,1400,11,1.084507,6,1.400000,0.491803,0.428571,0.229508,0.846154,...,1.114250,0.094162,0.219711,0.400000,0.857143,1.000000,0.548387,0.578947,0.540984,0.577038
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86768,2026,3397,118,0.885057,-10,1.400000,0.455882,0.285714,0.308824,0.900000,...,1.235795,-0.142045,0.099432,0.290323,1.285714,0.473684,0.243243,0.645161,0.500000,0.531768
86769,2026,3438,118,0.987952,-1,0.571429,0.500000,0.266667,0.250000,0.720000,...,1.257576,-0.015152,0.060606,0.500000,3.750000,0.833333,0.300000,0.578947,0.533333,0.577465
86770,2026,3332,118,0.985714,-1,0.857143,0.490566,0.250000,0.301887,0.812500,...,1.227209,-0.017532,0.105189,0.500000,2.166667,1.000000,0.333333,0.625000,0.528302,0.574617
86771,2026,3153,118,0.508475,-58,0.181818,0.415094,0.181818,0.207547,0.583333,...,1.981195,-0.973808,0.033580,0.409091,4.500000,0.346154,0.193548,0.633333,0.433962,0.471995


# TODO

In [19]:
# # Get a cumulative total of wins and losses
# AllRegDetail = AllRegDetail.sort_values(['Season','TeamID','DayNum'])
# AllRegDetail['Wins'] = AllRegDetail.groupby(['Season','TeamID'])['Win'].cumsum()
# AllRegDetail['Losses'] = AllRegDetail.groupby(['Season','TeamID'])['Loss'].cumsum()
# 
# RegSeasonFeatures['W/L'] = df['Wins'] / df['Losses']  # Win/Loss ratio